In [25]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
cwd = Path().resolve()

if cwd.name == 'agents':
    project_root = cwd.parent.parent
elif cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
# react agent basic
# Please use these langchain versions for this cell
# pip install langchain==0.3.18 langchain-community==0.3.17 langchain-core==0.3.45

from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent # uses react prompt under the hood
from langchain_community.tools import TavilySearchResults

llm = ChatOpenAI(model = "gpt-3.5-turbo")

search_tool = TavilySearchResults(search_depth="basic")

@tool
def get_system_time(format: str = "%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(format)

tools = [search_tool, get_system_time]
agent = initialize_agent(tools=tools, llm=llm, agent="zero-shot-react-description", verbose=True)

# agent.invoke("Give me a funny tweet about today's weather in Bangalore")
agent.invoke("When was spaceX's last launch and how many days ago was it that from this instant?")

TypeError: create_agent() got an unexpected keyword argument 'agent'

In [ ]:
# reflection agent

from typing import List, Sequence
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import END, MessageGraph

llm = ChatOpenAI(model="gpt-3.5-turbo")

#generation_chain
generation_prompt = ChatPromptTemplate.from_messages([
    (
        "system", "You are a twitter techie influencer assitant tasked with writing excellent twitter posts. Genrate the best twitter post possible for the users request. If the user provide critique, response with a revised version of your previous attempts."
    ),
    MessagesPlaceholder(variable_name="messages") # chat history
])
generation_chain = generation_prompt | llm

#reflection_chain
reflection_prompt = ChatPromptTemplate.from_messages([
    (
        "system", "Your are a viral twitter influencer grading a tweet. Generate critique and recommendations for the user's tweet. Always provide detailed recommendations, including requests for length, virality, style, etc."
    ),
    MessagesPlaceholder(variable_name="messages")
])
reflection_chain = reflection_prompt | llm

# MessageGraph - a class langgraph provides to orchestrate the flow of messages between different nodes
# if the app requires complex state management we have stategraph

graph = MessageGraph()

REFLECT = "reflect"
GENERATE = "generate"

def generate_node(state):
    return generation_chain.invoke({
        "messages": state
    })

def reflect_node(state):
    response = reflection_chain.invoke({
        "messages": state
    })
    return [HumanMessage(content=response.content)]

graph.add_node(GENERATE, generate_node)
graph.add_node(REFLECT, reflect_node)

graph.set_entry_point(GENERATE)

def should_continue(state):
    if len(state) > 4:
        return END
    return REFLECT

graph.add_conditional_edges(GENERATE, should_continue)
graph.add_edge(REFLECT,GENERATE)
app = graph.compile()
print(app.get_graph().draw_mermaid())
app.get_graph().print_ascii()

app.invoke(HumanMessage(content="Write a tweet about Python programming."))

In [ ]:
# structured outputs from LLMs - .with_structured_output()

# 1. use pydantic models
from pydantic import BaseModel, Field
class Country(BaseModel):
    """Information about a country"""
    name: str = Field(description="name of the country")
    capital: str = Field(description="capital city of the country")
    population: int = Field(description="population of the country")

structured_llm = llm.with_structured_output(Country)
structured_llm
structured_llm.invoke("What is the capital of France?")

# 2. typedDict
from typing_extensions import TypedDict, Annotated
from typing import Optional

class Joke(TypedDict):
    """Joke to tell user."""
    setup: Annotated[str, ..., "The setup of the joke"]
    punchline: Annotated[str, ..., "The punchline of the joke"]
    rating: Annotated[Optional[int], None, "The rating of the joke, from 1 to 10"]

structured_llm = llm.with_structured_output(Joke)
structured_llm.invoke("Tell me a joke about Python programming!")

# 3. json schema
json_schema = {
    "title": "joke",
    "description": "A joke to tell the user",
    "type": "object",
    "properties": {
        "setup": {
            "type": "string",
            "description": "The setup of the joke"
        },
        "punchline": {
            "type": "string",
            "description": "The punchline of the joke"
        },
        "rating": {
            "type": "integer",
            "description": "The rating of the joke, from 1 to 10",
            "default": None
        }
    },
    "required": ["setup", "punchline"]
}
structured_llm = llm.with_structured_output(json_schema)

In [ ]:
# StateGraph
from typing import TypedDict
from langgraph.graph import StateGraph

class SimpleState(TypedDict):
    count: int

def increment_node(state: SimpleState)->SimpleState:
    return {"count": state["count"] + 1}

def should_continue(state):
    if(state["count"]<5):
        return "continue"
    else:
        return "stop"

graph = StateGraph(SimpleState)

graph.add_node("increment", increment_node)
graph.add_conditional_edges("increment", should_continue, {"continue": "increment", "stop": END})
graph.set_entry_point("increment")

app = graph.compile()

state = {
    "count": 0
}

res = app.invoke(state)

In [ ]:
# manual State transformation
class SimpleState(TypedDict):
    count: int
    sum: int
    history: List[int]

def manual_increment(state: SimpleState) -> SimpleState:
    new_count = state["count"] + 1
    return {
        "count": new_count,
        "sum": state["sum"] + new_count,
        "history": state["history"] + [new_count]
    }

# Declarative Annotated State transformation
class SimpleState(TypedDict):
    count: int
    sum: Annotated[int, operator.add]
    history: Annotated[List[int], operator.concat]

def annotated_increment(state: SimpleState) -> SimpleState:
    new_count = state["count"] + 1

    return {
        "count": new_count,
        "sum": new_count,
        "history": [new_count]
    }